# V3 Model Evaluator

This notebook evaluates one trained v3 RL policy against fixed-chain and per-circuit baselines using the current feature-batch-aware environment setup.

In [ ]:
from pathlib import Path
import os
import sys
import textwrap
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from stable_baselines3 import A2C, DQN, PPO

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "dbs" / "combined_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

V3_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "two_opt_chain_rl_v3"
if str(V3_DIR) not in sys.path:
    sys.path.insert(0, str(V3_DIR))

from data_processing import feature_batch_names, split_summary_frame
from train import build_env, reward_splits
from evaluation import evaluate_model_on_frame, summarize_selected_frame
from rewards import (
    SUPPORTED_REWARD_TYPES,
    single_step_reward,
    incremental_step_reward,
    compute_final_correction,
)

sns.set_theme(style="ticks", context="paper")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

ALGORITHM_CLASSES = {
    "ppo": PPO,
    "a2c": A2C,
    "dqn": DQN,
}

FEATURE_BATCH_ALIASES = {
    "all_features": "plus_all_features",
    "plus_all_no_history": "plus_all_features",
}

TRAINING_LOG_PATH = V3_DIR / "training_runs.csv"
MODEL_PATHS = None
MODEL_ALGORITHM = "ppo"
SELECTED_ALGORITHM_SEEDS = None
FEATURE_BATCH = "all_features"
BASELINE_REWARD_TYPE = "mixed_weighted"
RUNTIME_WEIGHT = 0.1
TWO_QUBIT_WEIGHT = 1.0
DEPTH_WEIGHT = 0.1
FINAL_REWARD_WEIGHT = 1.0
MODEL_METHOD_NAME = None
EVAL_SPLIT = "validation"
MODEL_SELECTION_METRIC = f"{'validation' if EVAL_SPLIT == 'train' else EVAL_SPLIT}_utility"
SAVE_FIGURES = True
OUTPUT_DIR = V3_DIR / "model_evaluator_figures"


## Resolve Model

You can either set `MODEL_PATH` directly or leave it as `None` and let the notebook resolve the best matching row from `training_runs.csv`.

In [ ]:
FEATURE_BATCH = FEATURE_BATCH_ALIASES.get(FEATURE_BATCH, FEATURE_BATCH)

if MODEL_ALGORITHM.lower() not in ALGORITHM_CLASSES:
    raise ValueError(f"Unsupported MODEL_ALGORITHM: {MODEL_ALGORITHM}")

if FEATURE_BATCH not in feature_batch_names():
    raise ValueError(
        f"Unsupported FEATURE_BATCH: {FEATURE_BATCH}. Supported values: {feature_batch_names()}"
    )

if BASELINE_REWARD_TYPE not in SUPPORTED_REWARD_TYPES:
    raise ValueError(
        f"Unsupported BASELINE_REWARD_TYPE: {BASELINE_REWARD_TYPE}. Supported values: {SUPPORTED_REWARD_TYPES}"
    )

if EVAL_SPLIT not in {"train", "validation", "test"}:
    raise ValueError('EVAL_SPLIT must be one of: "train", "validation", "test".')

numeric_training_columns = [
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]


def load_training_log(path):
    if not path.exists():
        return pd.DataFrame()

    frame = pd.read_csv(path)
    for column in numeric_training_columns:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    if "algorithm_seed" in frame.columns:
        frame["algorithm_seed"] = pd.to_numeric(frame["algorithm_seed"], errors="coerce").fillna(42).astype(int)
    if "feature_batch" in frame.columns:
        frame["feature_batch"] = frame["feature_batch"].fillna("original")
    return frame


def close_match(series, value, atol=1e-9):
    numeric = pd.to_numeric(series, errors="coerce")
    return np.isclose(numeric, float(value), atol=atol, rtol=0.0, equal_nan=False)


training_log = load_training_log(TRAINING_LOG_PATH)
selected_training_rows = pd.DataFrame()

if MODEL_PATHS is not None:
    if isinstance(MODEL_PATHS, (str, Path)):
        requested_model_paths = [MODEL_PATHS]
    else:
        requested_model_paths = list(MODEL_PATHS)

    resolved_model_paths = []
    for raw_path in requested_model_paths:
        path = Path(raw_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Model file not found: {path}")
        resolved_model_paths.append(path)

    if training_log.empty:
        selected_training_rows = pd.DataFrame(
            {
                "model_path": [str(path) for path in resolved_model_paths],
                "algorithm_type": [MODEL_ALGORITHM] * len(resolved_model_paths),
                "algorithm_seed": [None] * len(resolved_model_paths),
                "feature_batch": [FEATURE_BATCH] * len(resolved_model_paths),
                "reward_type": [BASELINE_REWARD_TYPE] * len(resolved_model_paths),
                "runtime_weight": [RUNTIME_WEIGHT] * len(resolved_model_paths),
                "two_qubit_weight": [TWO_QUBIT_WEIGHT] * len(resolved_model_paths),
                "depth_weight": [DEPTH_WEIGHT] * len(resolved_model_paths),
                "final_reward_weight": [FINAL_REWARD_WEIGHT] * len(resolved_model_paths),
            }
        )
    else:
        resolved_set = {str(path) for path in resolved_model_paths}
        file_name_set = {path.name for path in resolved_model_paths}
        selected_training_rows = training_log[
            training_log["model_path"].astype(str).isin(resolved_set)
            | training_log["model_path"].astype(str).map(lambda text: Path(text).name).isin(file_name_set)
        ].copy()
        if selected_training_rows.empty:
            raise ValueError(
                "MODEL_PATHS were provided, but no matching rows were found in training_runs.csv. "
                "Either add those runs to the training log or set the algorithm/reward metadata manually."
            )
else:
    if training_log.empty:
        raise FileNotFoundError(
            "MODEL_PATHS is None and training_runs.csv is unavailable, so the notebook cannot resolve models automatically."
        )

    filtered = training_log[
        (training_log["algorithm_type"].str.lower() == MODEL_ALGORITHM.lower())
        & (training_log["feature_batch"] == FEATURE_BATCH)
        & (training_log["reward_type"] == BASELINE_REWARD_TYPE)
        & close_match(training_log["runtime_weight"], RUNTIME_WEIGHT)
        & close_match(training_log["two_qubit_weight"], TWO_QUBIT_WEIGHT)
        & close_match(training_log["depth_weight"], DEPTH_WEIGHT)
        & close_match(training_log["final_reward_weight"], FINAL_REWARD_WEIGHT)
    ].copy()

    if SELECTED_ALGORITHM_SEEDS is not None:
        requested_seeds = {int(seed) for seed in SELECTED_ALGORITHM_SEEDS}
        filtered = filtered[filtered["algorithm_seed"].isin(requested_seeds)].copy()

    if filtered.empty:
        raise ValueError(
            "No training log rows matched the requested model settings. "
            "Either adjust the filters or set MODEL_PATHS directly."
        )

    sort_columns = [
        column for column in [MODEL_SELECTION_METRIC, "validation_utility", "test_utility"]
        if column in filtered.columns
    ]
    selected_training_rows = (
        filtered.sort_values(sort_columns, ascending=False)
        .drop_duplicates(subset=["algorithm_seed"], keep="first")
        .sort_values("algorithm_seed")
        .reset_index(drop=True)
    )

if selected_training_rows.empty:
    raise ValueError("No model rows were selected for evaluation.")

selected_training_rows["model_path"] = selected_training_rows["model_path"].map(lambda path: str(Path(path).expanduser().resolve()))
missing_model_paths = [path for path in selected_training_rows["model_path"] if not Path(path).exists()]
if missing_model_paths:
    raise FileNotFoundError(f"Model file(s) not found: {missing_model_paths}")

reference_row = selected_training_rows.iloc[0].copy()
MODEL_ALGORITHM = str(reference_row["algorithm_type"]).lower()
FEATURE_BATCH = str(reference_row.get("feature_batch", FEATURE_BATCH))
BASELINE_REWARD_TYPE = str(reference_row["reward_type"])
RUNTIME_WEIGHT = float(reference_row["runtime_weight"])
TWO_QUBIT_WEIGHT = float(reference_row["two_qubit_weight"])
DEPTH_WEIGHT = float(reference_row["depth_weight"])
FINAL_REWARD_WEIGHT = float(reference_row["final_reward_weight"])
selected_algorithm_seeds = selected_training_rows["algorithm_seed"].dropna().astype(int).tolist()

model_method_name = MODEL_METHOD_NAME or f"{MODEL_ALGORITHM.upper()} agent"

selection_summary = pd.DataFrame(
    {
        "model_algorithm": [MODEL_ALGORITHM],
        "feature_batch": [FEATURE_BATCH],
        "evaluation_split": [EVAL_SPLIT],
        "selection_metric": [MODEL_SELECTION_METRIC],
        "baseline_reward_type": [BASELINE_REWARD_TYPE],
        "runtime_weight": [RUNTIME_WEIGHT],
        "two_qubit_weight": [TWO_QUBIT_WEIGHT],
        "depth_weight": [DEPTH_WEIGHT],
        "final_reward_weight": [FINAL_REWARD_WEIGHT],
        "selected_model_count": [len(selected_training_rows)],
        "selected_seeds": [", ".join(str(seed) for seed in selected_algorithm_seeds) if selected_algorithm_seeds else "(unlogged)"],
    }
)

display(selection_summary)
display(selected_training_rows.reset_index(drop=True))


## Prepare Datasets

This uses the same reward and feature-batch pipeline as `train.py`, so the evaluator stays aligned with the current v3 environment.

In [ ]:
bundle = reward_splits(
    runtime_weight=RUNTIME_WEIGHT,
    reward_type=BASELINE_REWARD_TYPE,
    two_qubit_weight=TWO_QUBIT_WEIGHT,
    depth_weight=DEPTH_WEIGHT,
    save_splits=False,
    feature_batch=FEATURE_BATCH,
)

optimizer_names = bundle["optimizer_names"]
category_names = bundle["category_names"]
metric_scale = bundle["metric_scale"]
observation_feature_scale = bundle["observation_feature_scale"]
static_feature_columns = bundle["static_feature_columns"]
dynamic_feature_columns = bundle["dynamic_feature_columns"]
include_category = bundle["feature_batch_config"]["include_category"]
train_df = bundle["train_df"].copy()
validation_df = bundle["validation_df"].copy()
test_df = bundle["test_df"].copy()


def add_episode_columns(frame):
    df = frame.copy()
    row_lookup = {
        (row.original_circuit, row.chain_name): row._asdict()
        for row in df.itertuples(index=False)
    }

    episode_rewards = []
    for row in df.itertuples(index=False):
        row_dict = row._asdict()

        if "__" not in row.chain_name:
            reward = float(
                single_step_reward(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                compute_final_correction(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    final_reward_weight=FINAL_REWARD_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
        else:
            first_optimizer = row.chain_name.split("__", 1)[0]
            first_row = row_lookup[(row.original_circuit, first_optimizer)]
            reward = float(
                single_step_reward(
                    first_row,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                incremental_step_reward(
                    row_dict,
                    first_row,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                compute_final_correction(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    final_reward_weight=FINAL_REWARD_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )

        episode_rewards.append(reward)

    df["episode_reward_if_selected"] = episode_rewards
    df["episode_total_runtime_seconds"] = pd.to_numeric(df["total_runtime_seconds"], errors="coerce")
    df["episode_runtime_score"] = pd.to_numeric(df["total_runtime_score"], errors="coerce")
    return df

train_df = add_episode_columns(train_df)
validation_df = add_episode_columns(validation_df)
test_df = add_episode_columns(test_df)

eval_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}
eval_df = eval_frames[EVAL_SPLIT].copy()

feature_batch_summary = pd.DataFrame(
    {
        "feature_batch": [FEATURE_BATCH],
        "include_category": [include_category],
        "static_feature_columns": [", ".join(static_feature_columns) or "(none)"],
        "dynamic_feature_columns": [", ".join(dynamic_feature_columns) or "(none)"],
        "description": [bundle["feature_batch_config"]["description"]],
        "eval_circuits": [eval_df["original_circuit"].nunique()],
    }
)

display(split_summary_frame({
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}))
display(feature_batch_summary)


## Evaluate The RL Policy

In [ ]:
rl_selected_frames = []
rl_summary_rows = []
evaluation_check_rows = []

for selected_row in selected_training_rows.itertuples(index=False):
    model_path = Path(selected_row.model_path)
    model_seed = None if pd.isna(selected_row.algorithm_seed) else int(selected_row.algorithm_seed)
    model = ALGORITHM_CLASSES[MODEL_ALGORITHM].load(str(model_path))

    seed_selected = evaluate_model_on_frame(
        model,
        eval_df,
        build_env,
        optimizer_names,
        category_names,
        metric_scale,
        observation_feature_scale,
        static_feature_columns,
        dynamic_feature_columns,
        include_category,
        method_name=model_method_name,
        reward_type=BASELINE_REWARD_TYPE,
        runtime_weight=RUNTIME_WEIGHT,
        two_qubit_weight=TWO_QUBIT_WEIGHT,
        depth_weight=DEPTH_WEIGHT,
        final_reward_weight=FINAL_REWARD_WEIGHT,
        verbose=False
    )

    seed_selected["model_path"] = str(model_path)
    seed_selected["algorithm_seed"] = model_seed
    seed_selected["model_eval_id"] = f"seed_{model_seed}" if model_seed is not None else model_path.stem

    assert len(seed_selected) == eval_df["original_circuit"].nunique()
    assert seed_selected["selected_chain"].notna().all()

    seed_summary = summarize_selected_frame(seed_selected)
    rl_summary_rows.append(
        {
            "algorithm_seed": model_seed,
            "model_path": str(model_path),
            "avg_reward_utility": seed_summary["utility"],
            "avg_pct_reduction": seed_summary["avg_two_qubit_gate_reduction_pct"],
            "avg_depth_reduction": seed_summary["avg_depth_reduction_pct"],
            "avg_runtime_seconds": seed_summary["avg_runtime_seconds"],
            "avg_runtime_score": seed_summary["avg_runtime_score"],
        }
    )
    evaluation_check_rows.append(
        {
            "algorithm_seed": model_seed,
            "model_path": str(model_path),
            "selected_circuits": len(seed_selected),
            "expected_circuits": eval_df["original_circuit"].nunique(),
            "selected_original_rate": float((seed_selected["selected_chain"] == "__original__").mean()),
            "selected_single_chain_rate": float((~seed_selected["selected_chain"].str.contains("__", regex=False)).mean()),
            "selected_double_chain_rate": float((seed_selected["selected_chain"].str.contains("__", regex=False)).mean()),
        }
    )
    rl_selected_frames.append(seed_selected)

rl_selected = pd.concat(rl_selected_frames, ignore_index=True)
rl_seed_summaries = pd.DataFrame(rl_summary_rows)
evaluation_checks = pd.DataFrame(evaluation_check_rows)

rl_summary = pd.DataFrame(
    [
        {
            "method": model_method_name,
            "circuits": eval_df["original_circuit"].nunique(),
            "evaluated_models": len(rl_seed_summaries),
            "avg_reward_utility": float(rl_seed_summaries["avg_reward_utility"].mean()),
            "avg_pct_reduction": float(rl_seed_summaries["avg_pct_reduction"].mean()),
            "avg_depth_reduction": float(rl_seed_summaries["avg_depth_reduction"].mean()),
            "avg_runtime_seconds": float(rl_seed_summaries["avg_runtime_seconds"].mean()),
            "avg_runtime_score": float(rl_seed_summaries["avg_runtime_score"].mean()),
        }
    ]
)

display(evaluation_checks)
display(rl_seed_summaries)
display(rl_summary)
display(rl_selected.head())


## Baselines And Summary Tables

In [ ]:
def fixed_chain_rows(frame, chain_name, method_name):
    rows = frame[frame["chain_name"] == chain_name].copy()
    rows["method"] = method_name
    rows["selected_chain"] = chain_name
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows


def no_optimization_row(frame, circuit):
    row = frame[frame["original_circuit"] == circuit].iloc[0].copy()
    row["chain_name"] = "__original__"
    row["selected_chain"] = "__original__"
    row["invalid_action"] = False
    row["final_num_qubits"] = row["initial_num_qubits"]
    row["final_depth"] = row["initial_depth"]
    row["final_two_qubit_gates"] = row["initial_two_qubit_gates"]
    row["final_total_gates"] = row["initial_total_gates"]
    row["final_two_qubit_depth"] = row["initial_two_qubit_depth"]
    row["total_runtime_seconds"] = 0.0
    row["total_runtime_score"] = 0.0
    row["total_two_qubit_gate_reduction"] = 0.0
    row["total_two_qubit_gate_reduction_pct"] = 0.0
    row["total_depth_reduction"] = 0.0
    row["total_depth_reduction_pct"] = 0.0
    row["episode_reward_if_selected"] = 0.0
    row["two_qubit_gate_reduction"] = 0.0
    row["two_qubit_gate_reduction_pct"] = 0.0
    row["depth_reduction"] = 0.0
    row["depth_reduction_pct"] = 0.0
    row["current_step_two_qubit_gate_reduction"] = 0.0
    row["current_step_two_qubit_gate_reduction_pct"] = 0.0
    row["current_step_depth_reduction"] = 0.0
    row["current_step_depth_reduction_pct"] = 0.0
    row["runtime_score"] = 0.0
    row["local_step_reward"] = 0.0
    row["utility_reward"] = 0.0
    return row


def best_possible_depth_rows(frame, method_name):
    best_rows = frame.loc[frame.groupby("original_circuit")["depth_reduction_pct"].idxmax()].copy()
    chosen_rows = []
    for circuit in sorted(frame["original_circuit"].unique()):
        best_row = best_rows[best_rows["original_circuit"] == circuit].iloc[0]
        original_row = no_optimization_row(frame, circuit)
        if float(original_row["depth_reduction_pct"]) >= float(best_row["depth_reduction_pct"]):
            chosen_rows.append(original_row)
        else:
            chosen_rows.append(best_row.copy())
    rows = pd.DataFrame(chosen_rows)
    rows["method"] = method_name
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows


def best_possible_twoq_rows(frame, method_name):
    best_rows = frame.loc[
        frame.groupby("original_circuit")["total_two_qubit_gate_reduction_pct"].idxmax()
    ].copy()
    chosen_rows = []
    for circuit in sorted(frame["original_circuit"].unique()):
        best_row = best_rows[best_rows["original_circuit"] == circuit].iloc[0]
        original_row = no_optimization_row(frame, circuit)
        if float(original_row["total_two_qubit_gate_reduction_pct"]) >= float(best_row["total_two_qubit_gate_reduction_pct"]):
            chosen_rows.append(original_row)
        else:
            chosen_rows.append(best_row.copy())
    rows = pd.DataFrame(chosen_rows)
    rows["method"] = method_name
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows


def summarize_selected(selected_rows, method_name):
    summary = summarize_selected_frame(selected_rows)
    return {
        "method": method_name,
        "circuits": len(selected_rows),
        "avg_reward_utility": summary["utility"],
        "avg_pct_reduction": summary["avg_two_qubit_gate_reduction_pct"],
        "avg_depth_reduction": summary["avg_depth_reduction_pct"],
        "avg_runtime_seconds": summary["avg_runtime_seconds"],
        "avg_runtime_score": summary["avg_runtime_score"],
    }


single_chains = sorted(name for name in train_df["chain_name"].unique() if "__" not in name)
double_chains = sorted(name for name in train_df["chain_name"].unique() if "__" in name)

best_fixed_reward_chain = (
    train_df.groupby("chain_name")["episode_reward_if_selected"].mean().sort_values(ascending=False).index[0]
)
best_single_optimizer_for_depth_reduction = (
    train_df[train_df["chain_name"].isin(single_chains)]
    .groupby("chain_name")["depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
best_single_optimizer_for_two_qubit_reduction = (
    train_df[train_df["chain_name"].isin(single_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_chain_for_depth_reduction = (
    train_df[train_df["chain_name"].isin(double_chains)]
    .groupby("chain_name")["depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
best_chain_for_two_qubit_reduction = (
    train_df[train_df["chain_name"].isin(double_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_reward_chain_label = f"Best fixed reward chain ({best_fixed_reward_chain})"
best_depth_chain_label = "Best depth chain"
best_twoq_chain_label = "Best 2Q chain"
best_depth_single_label = f"Best depth reduction single"
best_twoq_single_label = f"Best 2Q gate reduction single"
best_possible_depth_label = "Best possible depth reduction"
best_possible_twoq_label = "Best possible two-qubit gate reduction"

optimizer_display_labels = {
    "qiskit_standard": "Qiskit",
    "qiskit_ai": "Qiskit AI",
    "tket": "TKET",
    "wisq_rules": "WISQ Rules",
    "wisq_bqskit": "WISQ BQSKit",
}
optimizer_baseline_labels = {
    chain_name: optimizer_display_labels.get(chain_name, chain_name)
    for chain_name in single_chains
    if chain_name in optimizer_display_labels
}

baseline_reference = pd.DataFrame(
    [
        {"baseline": best_reward_chain_label, "chain": best_fixed_reward_chain},
        {"baseline": best_depth_chain_label, "chain": best_chain_for_depth_reduction},
        {"baseline": best_twoq_chain_label, "chain": best_chain_for_two_qubit_reduction},
        {"baseline": best_depth_single_label, "chain": best_single_optimizer_for_depth_reduction},
        {"baseline": best_twoq_single_label, "chain": best_single_optimizer_for_two_qubit_reduction},
        {"baseline": best_possible_depth_label, "chain": "per-circuit upper bound incl. No Optimization"},
        {"baseline": best_possible_twoq_label, "chain": "per-circuit upper bound incl. No Optimization"},
    ]
)
display(baseline_reference)

baseline_selected_frames = {
    best_reward_chain_label: fixed_chain_rows(eval_df, best_fixed_reward_chain, best_reward_chain_label),
    best_depth_chain_label: fixed_chain_rows(eval_df, best_chain_for_depth_reduction, best_depth_chain_label),
    best_twoq_chain_label: fixed_chain_rows(eval_df, best_chain_for_two_qubit_reduction, best_twoq_chain_label),
    best_depth_single_label: fixed_chain_rows(eval_df, best_single_optimizer_for_depth_reduction, best_depth_single_label),
    best_twoq_single_label: fixed_chain_rows(eval_df, best_single_optimizer_for_two_qubit_reduction, best_twoq_single_label),
    best_possible_depth_label: best_possible_depth_rows(eval_df, best_possible_depth_label),
    best_possible_twoq_label: best_possible_twoq_rows(eval_df, best_possible_twoq_label),
}

optimizer_selected_frames = {
    label: fixed_chain_rows(eval_df, chain_name, label)
    for chain_name, label in optimizer_baseline_labels.items()
}

baseline_plot_frames = {
    label: frame
    for label, frame in baseline_selected_frames.items()
    if label not in { best_possible_twoq_label, best_reward_chain_label, best_possible_depth_label, best_depth_single_label, best_depth_chain_label }
}

comparison_rows = [
    summarize_selected(frame, label)
    for label, frame in baseline_plot_frames.items()
]
comparison_rows.append(rl_summary.iloc[0].drop(labels=["evaluated_models"]).to_dict())
comparison = pd.DataFrame(comparison_rows).sort_values("avg_reward_utility", ascending=False).reset_index(drop=True)

display(comparison)


## Plots

In [ ]:
def prettify_label(name, wrap_width=24):
    if str(name).strip() == "__original__":
        return "No Optimization"
    name = str(name).replace("__", " --> ").replace("_", " ")
    replacements = {
        "qiskit": "Qiskit",
        "qiskit ai": "Qiskit AI",
        "wisq rules": "WISQ Rules",
        "wisq bqskit": "WISQ BQSKit",
        "tket": "TKET",
        "best fixed reward chain": "Best Fixed Reward Chain",
        "best depth reduction chain": "Best Depth Reduction Chain",
        "best 2q gate reduction chain": "Best 2Q Gate Reduction Chain",
        "best depth reduction single": "Best Depth Reduction ",
        "best 2q gate reduction single": "Best 2Q Single",
        "best possible depth reduction": "Best Possible Depth Reduction",
        "best possible two-qubit gate reduction": "Best Possible Two-Qubit Gate Reduction",
        "ppo agent": "RL Agent",
        "a2c agent": "A2C Agent",
        "dqn agent": "DQN Agent",
        "best depth chain": "Best Depth Chain",
        "best 2q chain": "Best 2Q Chain",
    }

    lowered = name.lower()
    for old, new in replacements.items():
        lowered = lowered.replace(old, new)
    pretty = lowered
    if wrap_width is None:
        return pretty
    return textwrap.fill(pretty, width=wrap_width)


# -----------------------------------------------------------------------------
# Font-size controls. Every figure font size used below is editable here.
# -----------------------------------------------------------------------------
# Heatmap only: selected_chain_counts_heatmap.pdf.
# Main title above the heatmap.
HEATMAP_TITLE_SIZE = 14
# X-axis name 'Selected optimizer chain' beneath the heatmap.
HEATMAP_X_AXIS_LABEL_SIZE = 12
# Y-axis name 'Method' beside the heatmap.
HEATMAP_Y_AXIS_LABEL_SIZE = 12
# Optimizer-chain names along the bottom of the heatmap.
HEATMAP_X_TICK_SIZE = 10
# Method names along the left side of the heatmap.
HEATMAP_Y_TICK_SIZE = 10
# Count printed inside each heatmap cell.
HEATMAP_CELL_VALUE_SIZE = 9
# Numbered tick values beside the heatmap colorbar.
HEATMAP_COLORBAR_TICK_SIZE = 10
# 'Number of test circuits' label beside the heatmap colorbar.
HEATMAP_COLORBAR_LABEL_SIZE = 12

# RL chain-usage chart only: model_chain_usage.pdf.
# Main title above the vertical chain-usage chart.
CHAIN_USAGE_TITLE_SIZE = 40
# X-axis name 'Selected optimizer chain' beneath the chain-usage chart.
CHAIN_USAGE_X_AXIS_LABEL_SIZE = 25
# Y-axis name 'Number of selections' beside the chain-usage chart.
CHAIN_USAGE_Y_AXIS_LABEL_SIZE = 20
# Optimizer-chain names along the bottom of the chain-usage chart.
CHAIN_USAGE_X_TICK_SIZE = 20
# Numbered selection-count ticks on the chain-usage y-axis.
CHAIN_USAGE_Y_TICK_SIZE = 20
# Selection count printed above each chain-usage bar.
CHAIN_USAGE_BAR_VALUE_SIZE = 15

# All standalone horizontal comparison charts made by horizontal_bar(...).
# This includes baseline-vs-RL and individual-optimizer PDF figures.
# Main title above each standalone horizontal comparison.
STANDALONE_TITLE_SIZE = 30
# Metric name beneath each standalone plot, such as average depth reduction.
STANDALONE_X_AXIS_LABEL_SIZE = 25
# Numbered metric ticks along each standalone x-axis.
STANDALONE_X_TICK_SIZE = 15
# Method or optimizer names along each standalone y-axis.
STANDALONE_METHOD_NAME_SIZE = 25
# Metric value printed inside each standalone horizontal bar.
STANDALONE_BAR_VALUE_SIZE = 20

# Combined 2x2 optimizer-comparison figure only.
# Main title centered above the complete 2x2 figure.
PANEL_MAIN_TITLE_SIZE = 25
# Subplot markers '(a)', '(b)', '(c)', and '(d)'.
PANEL_TAG_SIZE = 15
# Metric name beneath each subplot.
PANEL_X_AXIS_LABEL_SIZE = 20
# Numbered metric ticks along each subplot x-axis.
PANEL_X_TICK_SIZE = 10
# Optimizer and method names along each subplot y-axis.
PANEL_METHOD_NAME_SIZE = 20
# Metric value printed inside each subplot bar.
PANEL_BAR_VALUE_SIZE = 10

# Per-circuit utility-difference chart only: per_circuit_utility_difference.pdf.
# Main title above the per-circuit utility chart.
UTILITY_GAP_TITLE_SIZE = 14
# X-axis name beneath the per-circuit utility chart.
UTILITY_GAP_X_AXIS_LABEL_SIZE = 12
# Y-axis utility-difference name beside the per-circuit utility chart.
UTILITY_GAP_Y_AXIS_LABEL_SIZE = 12
# Circuit names along the bottom of the per-circuit utility chart.
UTILITY_GAP_X_TICK_SIZE = 8
# Numbered utility-difference ticks on the y-axis.
UTILITY_GAP_Y_TICK_SIZE = 10
# Publication palette: cool outer canvas with a warm, low-contrast plotting surface.
PAPER_FIGURE_COLOR = "#FFFFFF"
PAPER_AXES_COLOR = "#F7F4EA"
PAPER_GRID_COLOR = "#CDD5D9"
PAPER_PANEL_EDGE = "#657681"
PAPER_BAR_EDGE = "#40515F"
PAPER_BAR_ALPHA = 0.92
PAPER_LABEL_COLOR = "#24333D"
PAPER_BASE_BAR_COLOR = "#6F83A6"
PAPER_TRACK_COLOR = "#DFE5E7"
# Figure size and bar thickness knobs.
# Increase PAPER_FIG_WIDTH / PAPER_FIG_HEIGHT to give labels more room.
# Increase PAPER_BAR_HEIGHT to make the bars thicker.
PAPER_FIG_WIDTH = 9
PAPER_FIG_HEIGHT = 5.6
PAPER_BAR_HEIGHT = 0.58


def style_publication_axes(ax, grid_axis=None):
    """Apply the shared enclosed, paper-friendly style to one plotting axis."""
    ax.figure.set_facecolor(PAPER_FIGURE_COLOR)
    ax.set_facecolor(PAPER_AXES_COLOR)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", colors=PAPER_LABEL_COLOR, width=0.8)
    ax.xaxis.label.set_color(PAPER_LABEL_COLOR)
    ax.yaxis.label.set_color(PAPER_LABEL_COLOR)
    ax.title.set_color(PAPER_LABEL_COLOR)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(PAPER_PANEL_EDGE)
        spine.set_linewidth(0.8)

    ax.grid(False)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=PAPER_GRID_COLOR, linewidth=0.65, alpha=0.8)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=PAPER_GRID_COLOR, linewidth=0.65, alpha=0.8)


def save_publication_figure(fig, filename):
    if SAVE_FIGURES and filename is not None:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            OUTPUT_DIR / filename,
            bbox_inches="tight",
            dpi=300,
            facecolor=fig.get_facecolor(),
        )


# Heatmap: all methods in the evaluation split.
# plot_frames combines the RL agent with every baseline/oracle dataframe.
plot_frames = {model_method_name: rl_selected, **baseline_selected_frames}
selected_chain_counts = pd.concat(
    [frame[["method", "selected_chain"]] for frame in plot_frames.values()],
    ignore_index=True,
).groupby(["method", "selected_chain"]).size().reset_index(name="circuits")

selected_chain_grid = selected_chain_counts.pivot_table(
    index="method",
    columns="selected_chain",
    values="circuits",
    fill_value=0,
    aggfunc="sum",
)

method_order = [
    model_method_name,
    best_reward_chain_label,
    best_depth_chain_label,
    best_twoq_chain_label,
    best_depth_single_label,
    best_twoq_single_label,
    best_possible_depth_label,
    best_possible_twoq_label,
]
selected_chain_grid = selected_chain_grid.loc[
    [method for method in method_order if method in selected_chain_grid.index],
    selected_chain_grid.sum(axis=0).sort_values(ascending=False).index,
]

# This heatmap shows how often each method chose each chain.
selected_chain_grid_pretty = selected_chain_grid.copy()
selected_chain_grid_pretty.index = [prettify_label(name, wrap_width=28) for name in selected_chain_grid_pretty.index]
selected_chain_grid_pretty.columns = [prettify_label(name, wrap_width=18) for name in selected_chain_grid_pretty.columns]

display(selected_chain_grid_pretty)

heatmap_width = max(PAPER_FIG_WIDTH, 0.52 * len(selected_chain_grid_pretty.columns) + 3.2)
heatmap_height = max(PAPER_FIG_HEIGHT, 0.42 * len(selected_chain_grid_pretty.index) + 2.6)
fig, ax = plt.subplots(figsize=(heatmap_width, heatmap_height), facecolor=PAPER_FIGURE_COLOR)
style_publication_axes(ax)
sns.heatmap(
    selected_chain_grid_pretty,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#355F78", as_cmap=True),
    linewidths=0.5,
    linecolor=PAPER_AXES_COLOR,
    cbar_kws={"label": f"Number of {EVAL_SPLIT} circuits", "shrink": 0.9, "pad": 0.02},
    annot_kws={"size": HEATMAP_CELL_VALUE_SIZE},
    ax=ax,
)
ax.set_xlabel("Selected optimizer chain", fontsize=HEATMAP_X_AXIS_LABEL_SIZE, labelpad=10)
ax.set_ylabel("Method", fontsize=HEATMAP_Y_AXIS_LABEL_SIZE, labelpad=10)
ax.set_title("Optimizer-Chain Selection Frequency by Method", fontsize=HEATMAP_TITLE_SIZE, pad=9, weight="semibold")
ax.tick_params(axis="x", labelsize=HEATMAP_X_TICK_SIZE)
ax.tick_params(axis="y", labelsize=HEATMAP_Y_TICK_SIZE)
plt.setp(ax.get_xticklabels(), rotation=55, ha="right", rotation_mode="anchor")
plt.setp(ax.get_yticklabels(), rotation=0)
colorbar = ax.collections[0].colorbar
colorbar.ax.set_facecolor(PAPER_FIGURE_COLOR)
colorbar.ax.tick_params(labelsize=HEATMAP_COLORBAR_TICK_SIZE, colors=PAPER_LABEL_COLOR)
colorbar.set_label(f"Number of {EVAL_SPLIT} circuits", fontsize=HEATMAP_COLORBAR_LABEL_SIZE)
colorbar.outline.set_edgecolor(PAPER_PANEL_EDGE)
fig.tight_layout(pad=0.8)
save_publication_figure(fig, "selected_chain_counts_heatmap.pdf")
plt.show()
plt.close(fig)

# This vertical bar chart shows raw RL agent chain usage only.
model_chain_counts = (
    rl_selected["selected_chain"]
    .value_counts()
    .rename_axis("selected_chain")
    .reset_index(name="times_selected")
    .sort_values("times_selected", ascending=True)
)
model_chain_counts["display_chain"] = model_chain_counts["selected_chain"].apply(
    lambda name: prettify_label(name, wrap_width=None)
)

model_chain_counts = model_chain_counts.sort_values("times_selected", ascending=False).reset_index(drop=True)

usage_width = max(PAPER_FIG_WIDTH + 1.0, 0.72 * len(model_chain_counts) + 2.4)
fig, ax = plt.subplots(figsize=(usage_width, PAPER_FIG_HEIGHT), facecolor=PAPER_FIGURE_COLOR)
style_publication_axes(ax, grid_axis="y")
bars = ax.bar(
    model_chain_counts["display_chain"],
    model_chain_counts["times_selected"],
    color=PAPER_BASE_BAR_COLOR,
    edgecolor=PAPER_BAR_EDGE,
    linewidth=0.9,
    alpha=PAPER_BAR_ALPHA,
    width=0.68,
)

max_count = float(model_chain_counts["times_selected"].max()) if not model_chain_counts.empty else 1.0
ax.set_xlabel("Selected optimizer chain", fontsize=CHAIN_USAGE_X_AXIS_LABEL_SIZE, labelpad=8)
ax.set_ylabel("Number of selections", fontsize=CHAIN_USAGE_Y_AXIS_LABEL_SIZE, labelpad=8)
ax.set_title("RL Agent Optimizer-Chain Usage on the Test Split", fontsize=CHAIN_USAGE_TITLE_SIZE, pad=10, weight="semibold")
ax.tick_params(axis="x", labelsize=CHAIN_USAGE_X_TICK_SIZE, rotation=38)
ax.tick_params(axis="y", labelsize=CHAIN_USAGE_Y_TICK_SIZE)
plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

value_pad = max(0.02 * max_count, 0.15)
for bar, value in zip(bars, model_chain_counts["times_selected"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + value_pad,
        f"{int(value)}",
        ha="center",
        va="bottom",
        fontsize=CHAIN_USAGE_BAR_VALUE_SIZE,
        color=PAPER_LABEL_COLOR,
        fontweight="bold"
    )

ax.set_ylim(0, max_count * 1.18 if max_count > 0 else 1.0)
fig.tight_layout(pad=0.9)
save_publication_figure(fig, "model_chain_usage.pdf")
plt.show()
plt.close(fig)


# Standalone comparison plots all go through horizontal_bar(...).
# The dataframe you pass in controls which methods appear in the plot.
def horizontal_bar(data, x, xlabel, title, color=PAPER_BASE_BAR_COLOR, fmt="{:.3f}", filename=None, wrap_width=26):
    plot_data = data.sort_values(x, ascending=False).copy()
    plot_data["method_pretty"] = plot_data["method"].apply(
        lambda name: prettify_label(name, wrap_width=wrap_width)
    )

    figure_height = max(PAPER_FIG_HEIGHT, 0.48 * len(plot_data) + 1.7)
    fig, ax = plt.subplots(
        figsize=(PAPER_FIG_WIDTH, figure_height),
        facecolor=PAPER_FIGURE_COLOR,
    )
    style_publication_axes(ax, grid_axis="x")

    max_value = float(plot_data[x].max())
    min_value = float(plot_data[x].min())
    max_extent = max(abs(min_value), abs(max_value))
    if max_extent == 0:
        max_extent = 1.0

    baseline_values = [max_extent] * len(plot_data)
    if min_value < 0:
        left_offsets = [-max_extent] * len(plot_data)
        ax.barh(
            plot_data["method_pretty"],
            [2 * max_extent] * len(plot_data),
            left=left_offsets,
            color=PAPER_TRACK_COLOR,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.6,
            height=PAPER_BAR_HEIGHT,
        )
        bars = ax.barh(
            plot_data["method_pretty"],
            plot_data[x],
            color=color,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.8,
            height=PAPER_BAR_HEIGHT,
            alpha=PAPER_BAR_ALPHA,
        )
    else:
        ax.barh(
            plot_data["method_pretty"],
            baseline_values,
            color=PAPER_TRACK_COLOR,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.6,
            height=PAPER_BAR_HEIGHT,
        )
        bars = ax.barh(
            plot_data["method_pretty"],
            plot_data[x],
            color=color,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.8,
            height=PAPER_BAR_HEIGHT,
            alpha=PAPER_BAR_ALPHA,
        )

    ax.invert_yaxis()
    ax.set_title(title, fontsize=STANDALONE_TITLE_SIZE, pad=10, weight="semibold")
    ax.set_xlabel(xlabel, fontsize=STANDALONE_X_AXIS_LABEL_SIZE, labelpad=8)
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=STANDALONE_X_TICK_SIZE)
    ax.tick_params(axis="y", labelsize=STANDALONE_METHOD_NAME_SIZE, length=3, pad=6)

    value_pad = 0.012 * max_extent
    inside_threshold = 0.18 * max_extent

    for bar, value in zip(bars, plot_data[x]):
        y = bar.get_y() + bar.get_height() / 2
        if value >= 0:
            if value >= inside_threshold:
                value_x = value - value_pad
                value_ha = "right"
            else:
                value_x = value + value_pad
                value_ha = "left"
        else:
            if abs(value) >= inside_threshold:
                value_x = value + value_pad
                value_ha = "left"
            else:
                value_x = value - value_pad
                value_ha = "right"
        ax.text(value_x, y, fmt.format(value), va="center", ha=value_ha, fontsize=STANDALONE_BAR_VALUE_SIZE, color=PAPER_LABEL_COLOR)

    if min_value < 0:
        ax.set_xlim(-max_extent * 1.10, max_extent * 1.14)
        ax.axvline(0, color=PAPER_PANEL_EDGE, linewidth=0.9)
    else:
        ax.set_xlim(0, max_extent * 1.14)
    fig.subplots_adjust(left=0.34, right=0.97, bottom=0.12, top=0.90)

    save_publication_figure(fig, filename)
    plt.show()
    plt.close(fig)


# Baseline-vs-RL standalone figures.
# comparison is the dataframe that feeds these plots.
utility_plot = comparison.copy()
# These plots intentionally drop the oracle rows so the figure stays readable.
twoq_plot = comparison[~comparison["method"].isin([best_possible_depth_label])].copy()
depth_plot = comparison[~comparison["method"].isin([best_possible_twoq_label])].copy()
runtime_plot = comparison[~comparison["method"].isin([best_possible_depth_label, best_possible_twoq_label])].copy()
runtime_score_plot = runtime_plot.copy()

horizontal_bar(utility_plot, "avg_reward_utility", "Average utility reward", "Average Reward Utility by Method", filename="avg_reward_utility.pdf")
horizontal_bar(twoq_plot, "avg_pct_reduction", "Average two-qubit gate reduction (%)", "Average Two-Qubit Gate\nReduction by Method", color="#4F8A70", fmt="{:.1f}%", filename="avg_pct_reduction.pdf")
horizontal_bar(depth_plot, "avg_depth_reduction", "Average depth reduction (%)", "Average Depth Reduction by Method", color="#4F7F8C", fmt="{:.1f}%", filename="avg_depth_reduction.pdf")
horizontal_bar(runtime_plot, "avg_runtime_seconds", "Average runtime (s)", "Average Runtime by Method", color="#C47A3A", fmt="{:.2f}", filename="avg_runtime_seconds.pdf")
horizontal_bar(runtime_score_plot, "avg_runtime_score", "Average runtime score", "Average Runtime Score by Method", color="#B05A62", fmt="{:.3f}", filename="avg_runtime_score.pdf")


# 2x2 optimizer comparison panel.
# optimizer_comparison is the dataframe that feeds this figure.
def draw_panel_barh(ax, plot_data, column, xlabel, fmt, color, wrap_width=None):
    plot_data = plot_data.sort_values(column, ascending=False).copy()
    plot_data["method_pretty"] = plot_data["method"].apply(
        lambda name: prettify_label(name, wrap_width=wrap_width)
    )

    max_value = float(plot_data[column].max())
    min_value = float(plot_data[column].min())
    max_extent = max(abs(min_value), abs(max_value))
    if max_extent == 0:
        max_extent = 1.0

    style_publication_axes(ax, grid_axis="x")

    # Row positions are evenly spaced; physical spacing is controlled by PANEL_FIG_HEIGHT below.
    y_positions = np.arange(len(plot_data))
    bar_height = 0.55
    baseline_values = [max_extent] * len(plot_data)

    if min_value < 0:
        left_offsets = [-max_extent] * len(plot_data)
        ax.barh(
            y_positions,
            [2 * max_extent] * len(plot_data),
            left=left_offsets,
            color=PAPER_TRACK_COLOR,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.6,
            height=bar_height,
        )
        bars = ax.barh(
            y_positions,
            plot_data[column],
            color=color,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.8,
            height=bar_height,
            alpha=PAPER_BAR_ALPHA,
        )
    else:
        ax.barh(
            y_positions,
            baseline_values,
            color=PAPER_TRACK_COLOR,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.6,
            height=bar_height,
        )
        bars = ax.barh(
            y_positions,
            plot_data[column],
            color=color,
            edgecolor=PAPER_BAR_EDGE,
            linewidth=0.8,
            height=bar_height,
            alpha=PAPER_BAR_ALPHA,
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_data["method_pretty"], fontsize=PANEL_METHOD_NAME_SIZE)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=PANEL_X_AXIS_LABEL_SIZE, labelpad=6)
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=PANEL_X_TICK_SIZE)
    ax.tick_params(axis="y", length=3, pad=4)

    value_pad = 0.012 * max_extent
    inside_threshold = 0.20 * max_extent
    for bar, value in zip(bars, plot_data[column]):
        y = bar.get_y() + bar.get_height() / 2
        if value >= 0:
            if value >= inside_threshold:
                value_x = value - value_pad
                value_ha = "right"
            else:
                value_x = value + value_pad
                value_ha = "left"
        else:
            if abs(value) >= inside_threshold:
                value_x = value + value_pad
                value_ha = "left"
            else:
                value_x = value - value_pad
                value_ha = "right"
        ax.text(value_x, y, fmt.format(value), va="center", ha=value_ha, fontsize=PANEL_BAR_VALUE_SIZE, color=PAPER_LABEL_COLOR)

    if min_value < 0:
        ax.set_xlim(-max_extent * 1.08, max_extent * 1.14)
        ax.axvline(0, color=PAPER_PANEL_EDGE, linewidth=0.9)
    else:
        ax.set_xlim(0, max_extent * 1.14)

    # Add a small margin above and below the first and last rows.
    ax.set_ylim(y_positions[-1] + 0.7, -0.7)



# The four subplots come from optimizer_metric_specs below.
# Increase these values to give each enclosed panel more physical room.
PANEL_FIG_WIDTH = 18
PANEL_FIG_HEIGHT = 8.4

def plot_metric_panel(panel_specs, title, filename, wrap_width=None):
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(PANEL_FIG_WIDTH, PANEL_FIG_HEIGHT),
        facecolor=PAPER_FIGURE_COLOR,
    )
    axes = axes.flatten()

    for panel_index, (ax, (plot_data, column, xlabel, fmt, color)) in enumerate(zip(axes, panel_specs)):
        draw_panel_barh(ax, plot_data, column, xlabel, fmt, color, wrap_width=wrap_width)
        ax.set_title(f"({chr(97 + panel_index)})", loc="left", fontsize=PANEL_TAG_SIZE, weight="semibold", pad=5)

    panel_left = 0.23
    panel_right = 0.97
    panel_center = (panel_left + panel_right) / 2
    fig.suptitle(title, fontsize=PANEL_MAIN_TITLE_SIZE, y=0.975, x=panel_center, ha="center", weight="semibold")
    fig.subplots_adjust(left=panel_left, right=panel_right, bottom=0.08, top=0.91, wspace=0.48, hspace=0.38)
    save_publication_figure(fig, filename)
    plt.show()
    plt.close(fig)


# optimizer_comparison is the summary table used for the 2x2 panel and the standalone optimizer comparison plots.
optimizer_comparison_rows = [
    summarize_selected(frame, label)
    for label, frame in optimizer_selected_frames.items()
]
# Add the two best fixed-chain baselines to every optimizer-comparison plot.
# Because optimizer_metric_specs below reuses optimizer_comparison for all four panels,
# these chains appear in reward, 2Q, depth, and runtime subplots.
optimizer_comparison_rows.extend(
    [
        summarize_selected(baseline_selected_frames[best_depth_chain_label], best_depth_chain_label),
        summarize_selected(baseline_selected_frames[best_twoq_chain_label], best_twoq_chain_label),
    ]
)
optimizer_comparison_rows.append(rl_summary.iloc[0].drop(labels=["evaluated_models"]).to_dict())
optimizer_comparison = (
    pd.DataFrame(optimizer_comparison_rows)
    .sort_values("avg_reward_utility", ascending=False)
    .reset_index(drop=True)
)

display(optimizer_comparison)


# These are the standalone single-optimizer plots.
# Each tuple is: dataframe, column, x-label, title, color, value-format, filename.
optimizer_only_individual_specs = [
    (optimizer_comparison, "avg_reward_utility", "Average utility reward", "RL Agent vs Individual Optimizers: Reward Utility", PAPER_BASE_BAR_COLOR, "{:.3f}", "optimizer_only_avg_reward_utility.pdf"),
    (optimizer_comparison, "avg_pct_reduction", "Average two-qubit gate reduction (%)", "RL Agent vs Individual Optimizers:\nTwo-Qubit Gate Reduction", "#4F8A70", "{:.1f}%", "optimizer_only_avg_pct_reduction.pdf"),
    (optimizer_comparison, "avg_depth_reduction", "Average depth reduction (%)", "RL Agent vs Individual Optimizers: Depth Reduction", "#4F7F8C", "{:.1f}%", "optimizer_only_avg_depth_reduction.pdf"),
    (optimizer_comparison, "avg_runtime_seconds", "Average runtime (s)", "RL Agent vs Individual Optimizers: Runtime", "#C47A3A", "{:.2f}", "optimizer_only_avg_runtime_seconds.pdf"),
]

for plot_data, column, xlabel, title, color, fmt, filename in optimizer_only_individual_specs:
    horizontal_bar(
        plot_data,
        column,
        xlabel,
        title,
        color=color,
        fmt=fmt,
        filename=filename,
        wrap_width=18,
    )

# These are the four panels in the 2x2 optimizer comparison figure.
optimizer_metric_specs = [
    (optimizer_comparison, "avg_reward_utility", "Average utility reward", "{:.3f}", PAPER_BASE_BAR_COLOR),
    (optimizer_comparison, "avg_pct_reduction", "Average two-qubit gate reduction (%)", "{:.1f}%", "#4F8A70"),
    (optimizer_comparison, "avg_depth_reduction", "Average depth reduction (%)", "{:.1f}%", "#4F7F8C"),
    (optimizer_comparison, "avg_runtime_seconds", "Average runtime (s)", "{:.2f}", "#C47A3A"),
]

plot_metric_panel(
    optimizer_metric_specs,
    "RL Agent Compared with Individual Optimizers and Best Fixed Chains",
    "optimizer_only_comparison_panel.pdf",
    wrap_width=None,
)


## Per-Circuit Comparison Vs TKET --> WISQ BQSKit

In [ ]:
comparison_baseline_chain = "tket__wisq_bqskit"
if comparison_baseline_chain not in set(eval_df["chain_name"]):
    raise ValueError(f"Comparison baseline chain not found in evaluation frame: {comparison_baseline_chain}")

comparison_baseline_selected = fixed_chain_rows(
    eval_df,
    comparison_baseline_chain,
    comparison_baseline_chain,
)

rl_selected_by_circuit = (
    rl_selected.groupby("original_circuit", as_index=False)
    .agg(
        model_utility=("episode_reward", "mean"),
        model_selected_chain=("selected_chain", lambda values: values.value_counts().index[0]),
        model_selection_count=("selected_chain", "size"),
        model_twoq_reduction_pct=("total_two_qubit_gate_reduction_pct", "mean"),
        model_depth_reduction_pct=("total_depth_reduction_pct", "mean"),
        model_runtime_seconds=("episode_total_runtime_seconds", "mean"),
        model_runtime_score=("episode_runtime_score", "mean"),
    )
    .set_index("original_circuit")
)

baseline_by_circuit = comparison_baseline_selected.set_index("original_circuit")[
    [
        "selected_chain",
        "episode_reward",
        "total_two_qubit_gate_reduction_pct",
        "total_depth_reduction_pct",
        "episode_total_runtime_seconds",
        "episode_runtime_score",
    ]
].rename(
    columns={
        "selected_chain": "baseline_selected_chain",
        "episode_reward": "baseline_utility",
        "total_two_qubit_gate_reduction_pct": "baseline_twoq_reduction_pct",
        "total_depth_reduction_pct": "baseline_depth_reduction_pct",
        "episode_total_runtime_seconds": "baseline_runtime_seconds",
        "episode_runtime_score": "baseline_runtime_score",
    }
)

circuit_metric_compare = rl_selected_by_circuit.join(baseline_by_circuit, how="inner")
circuit_metric_compare["model_selected_chain_pretty"] = circuit_metric_compare[
    "model_selected_chain"
].map(lambda name: prettify_label(name, wrap_width=None))
circuit_metric_compare["baseline_selected_chain_pretty"] = circuit_metric_compare[
    "baseline_selected_chain"
].map(lambda name: prettify_label(name, wrap_width=None))

circuit_metric_compare["utility_diff"] = (
    circuit_metric_compare["model_utility"] - circuit_metric_compare["baseline_utility"]
)
circuit_metric_compare["twoq_reduction_diff"] = (
    circuit_metric_compare["model_twoq_reduction_pct"]
    - circuit_metric_compare["baseline_twoq_reduction_pct"]
)
circuit_metric_compare["depth_reduction_diff"] = (
    circuit_metric_compare["model_depth_reduction_pct"]
    - circuit_metric_compare["baseline_depth_reduction_pct"]
)
circuit_metric_compare["runtime_seconds_diff"] = (
    circuit_metric_compare["model_runtime_seconds"]
    - circuit_metric_compare["baseline_runtime_seconds"]
)
circuit_metric_compare["runtime_score_diff"] = (
    circuit_metric_compare["model_runtime_score"]
    - circuit_metric_compare["baseline_runtime_score"]
)

circuit_metric_compare = circuit_metric_compare.sort_values(
    ["utility_diff", "twoq_reduction_diff", "depth_reduction_diff"],
    ascending=[True, True, True],
)

comparison_display = circuit_metric_compare.reset_index()[
    [
        "original_circuit",
        "model_selected_chain",
        "model_selected_chain_pretty",
        "baseline_selected_chain",
        "baseline_selected_chain_pretty",
        "model_selection_count",
        "model_utility",
        "baseline_utility",
        "utility_diff",
        "model_twoq_reduction_pct",
        "baseline_twoq_reduction_pct",
        "twoq_reduction_diff",
        "model_depth_reduction_pct",
        "baseline_depth_reduction_pct",
        "depth_reduction_diff",
        "model_runtime_seconds",
        "baseline_runtime_seconds",
        "runtime_seconds_diff",
        "model_runtime_score",
        "baseline_runtime_score",
        "runtime_score_diff",
    ]
]

comparison_csv_path = OUTPUT_DIR / "per_circuit_model_vs_tket_wisq_bqskit.csv"
comparison_display.to_csv(comparison_csv_path, index=False)

print(
    f"Built per-circuit comparison for {len(comparison_display)} circuits. "
    f"Saved full table to {comparison_csv_path}"
)

display(comparison_display)

different_chain_display = comparison_display[
    comparison_display["model_selected_chain"] != comparison_display["baseline_selected_chain"]
].copy()
print(f"Circuits where the RL model chose a different chain than TKET --> WISQ BQSKit: {len(different_chain_display)}")
display(different_chain_display)

baseline_label_pretty = prettify_label(comparison_baseline_chain, wrap_width=None)
model_label_pretty = prettify_label(model_method_name, wrap_width=None)
metric_panel_specs = [
    {
        "column": "utility_diff",
        "title": "Utility difference",
        "ylabel": f"Utility ({model_label_pretty} - {baseline_label_pretty})",
        "positive_is_good": True,
    },
    {
        "column": "twoq_reduction_diff",
        "title": "2Q reduction difference",
        "ylabel": f"2Q reduction % ({model_label_pretty} - {baseline_label_pretty})",
        "positive_is_good": True,
    },
    {
        "column": "depth_reduction_diff",
        "title": "Depth reduction difference",
        "ylabel": f"Depth reduction % ({model_label_pretty} - {baseline_label_pretty})",
        "positive_is_good": True,
    },
    {
        "column": "runtime_seconds_diff",
        "title": "Runtime difference",
        "ylabel": f"Runtime seconds ({model_label_pretty} - {baseline_label_pretty})",
        "positive_is_good": False,
    },
]

panel_width = max(12.0, min(22.0, 0.25 * len(circuit_metric_compare) + 8.0))
fig, axes = plt.subplots(2, 2, figsize=(panel_width, 8.5), facecolor=PAPER_FIGURE_COLOR, sharex=True)
axes = axes.flatten()

for ax, spec in zip(axes, metric_panel_specs):
    style_publication_axes(ax, grid_axis="y")
    values = circuit_metric_compare[spec["column"]]
    positive_color = "#4F7F8C" if spec["positive_is_good"] else "#B05A62"
    negative_color = "#B05A62" if spec["positive_is_good"] else "#4F7F8C"
    ax.bar(
        circuit_metric_compare.index,
        values,
        color=np.where(values >= 0, positive_color, negative_color),
        edgecolor=PAPER_BAR_EDGE,
        linewidth=0.45,
        width=0.82,
    )
    ax.axhline(0, color=PAPER_PANEL_EDGE, linewidth=0.9)
    ax.set_title(spec["title"], fontsize=UTILITY_GAP_TITLE_SIZE, pad=8, weight="semibold")
    ax.set_ylabel(spec["ylabel"], fontsize=UTILITY_GAP_Y_AXIS_LABEL_SIZE, labelpad=7)
    ax.tick_params(axis="y", labelsize=UTILITY_GAP_Y_TICK_SIZE)
    ax.tick_params(axis="x", labelsize=UTILITY_GAP_X_TICK_SIZE, rotation=90)

for ax in axes[2:]:
    ax.set_xlabel(f"{EVAL_SPLIT.title()} circuit", fontsize=UTILITY_GAP_X_AXIS_LABEL_SIZE, labelpad=8)

fig.suptitle(
    f"Per-Circuit Metric Differences: {model_label_pretty} vs {baseline_label_pretty}",
    fontsize=UTILITY_GAP_TITLE_SIZE + 2,
    y=0.995,
    weight="semibold",
)
fig.tight_layout(pad=0.9, rect=(0, 0, 1, 0.96))
panel_pdf_path = OUTPUT_DIR / "per_circuit_metric_difference_panel.pdf"
panel_png_path = OUTPUT_DIR / "per_circuit_metric_difference_panel.png"
save_publication_figure(fig, panel_pdf_path.name)
save_publication_figure(fig, panel_png_path.name)
plt.show()
display(Image(filename=str(panel_png_path)))
print(f"Saved comparison figure to {panel_pdf_path} and {panel_png_path}")
plt.close(fig)




In [ ]:
print("\nFULL rl_selected_by_circuit:\n")
print(rl_selected_by_circuit.reset_index().to_string(index=False))

print("\nFULL baseline_by_circuit:\n")
print(baseline_by_circuit.reset_index().to_string(index=False))